# Phase C: CodeLlama 13B QLoRA — CCL Structural Similarity
## Lēthē / Force is Oblivion / Hegemonikón Research

CCL (Cognitive Control Language) の構造的パターンを QLoRA fine-tuning で獲得する。

| 項目 | 値 |
|:---|:---|
| **モデル** | CodeLlama-13b-hf (4bit QLoRA) |
| **データ** | 1000ペア (500 positive + 500 negative) |
| **GPU** | A100 (Colab Pro+) |
| **評価** | Spearman ρ, 偏ρ, F1, Accuracy |
| **比較対象** | Phase C-mini (CodeBERT ρ=0.963) |

### 実験仮説
- **P11'**: CodeLlama QLoRA > CodeBERT Phase C-mini (Δρ > 0)
- **P14**: L_Ξ 正則化 (λ>0) > L_Ξ なし (λ=0)

### データ準備
`phase_c_training_ccl.jsonl` を Google Drive にアップロードしてから実行すること。
```
Google Drive の任意の場所に phase_c_training_ccl.jsonl を配置
→ Cell 1 の DATA_PATH を合わせて変更
```

In [ ]:
# Cell 0: 環境確認 + インストール
!nvidia-smi
!pip install -q transformers peft bitsandbytes accelerate datasets trl scipy scikit-learn

In [ ]:
# Cell 1: Google Drive マウント + データ読込
from google.colab import drive
drive.mount('/content/drive')

# === データパス設定 ===
# Google Drive 内の phase_c_training_ccl.jsonl のパスを指定
DATA_PATH = '/content/drive/MyDrive/lethe/phase_c_training_ccl.jsonl'

import json
import numpy as np

def load_pairs(path):
    """JSONL から CCL ペアを読み込み"""
    pairs = []
    with open(path) as f:
        for line in f:
            d = json.loads(line)
            # CCL-only 形式: top-level に anchor_ccl / candidate_ccl
            anchor = d.get('anchor_ccl', '')
            candidate = d.get('candidate_ccl', '')
            # フォールバック: metadata 内にある場合
            if not anchor and 'metadata' in d:
                anchor = d['metadata'].get('anchor_ccl', '')
                candidate = d['metadata'].get('candidate_ccl', '')
            if anchor and candidate:
                pairs.append({
                    'anchor': anchor,
                    'candidate': candidate,
                    'label': d['label'],
                    'cosine_49d': d.get('cosine_49d', 0.0),
                    'pair_type': d.get('pair_type', 'unknown'),
                })
    return pairs

pairs = load_pairs(DATA_PATH)
pos = sum(1 for p in pairs if p['label'] == 1)
neg = len(pairs) - pos
print(f'\u2705 {len(pairs)} ペア読込 (positive: {pos}, negative: {neg})')
print(f'\nサンプル:')
print(f'  A: {pairs[0]["anchor"][:80]}...')
print(f'  B: {pairs[0]["candidate"][:80]}...')
print(f'  label: {pairs[0]["label"]}, type: {pairs[0]["pair_type"]}')

In [ ]:
# Cell 2: モデルロード (4bit 量子化)
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_NAME = 'codellama/CodeLlama-13b-hf'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'トークナイザロード中: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'モデルロード中 (4bit)...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,  # 回帰 (類似度スコア)
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.pad_token_id = tokenizer.pad_token_id

print(f'\u2705 ロード完了')
print(f'  パラメータ数: {model.num_parameters():,}')
print(f'  GPU メモリ: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# Cell 3: QLoRA 設定
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.SEQ_CLS,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 4: ベースライン測定 (fine-tune 前の素性能)
from scipy.stats import spearmanr
from sklearn.metrics import f1_score, accuracy_score

MAX_LEN = 512

def encode_pair(anchor, candidate):
    """CCL ペアをトークン化"""
    text = f'Structure A: {anchor}\n\nStructure B: {candidate}'
    return tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
        return_tensors='pt',
    )

def measure_rho(model, pairs, n_sample=200, seed=42):
    """モデルで類似度を予測し Spearman ρ を測定"""
    model.eval()
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(pairs), min(n_sample, len(pairs)), replace=False)
    preds, labels = [], []
    with torch.no_grad():
        for i, idx in enumerate(indices):
            p = pairs[idx]
            inputs = encode_pair(p['anchor'], p['candidate'])
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            out = model(**inputs)
            preds.append(out.logits.squeeze().item())
            labels.append(p['label'])
            if (i + 1) % 50 == 0:
                print(f'  [{i+1}/{len(indices)}]')
    rho, p_val = spearmanr(preds, labels)
    return rho, p_val

print('=== ベースライン測定 (fine-tune 前) ===')
baseline_rho, baseline_p = measure_rho(model, pairs)
print(f'\nベースライン ρ = {baseline_rho:.4f} (p = {baseline_p:.2e})')
print(f'比較: Phase C-mini (CodeBERT) ρ = 0.963')

In [ ]:
# Cell 5: データセット準備
from datasets import Dataset

def pairs_to_dataset(pairs):
    texts, labels, cosines = [], [], []
    for p in pairs:
        text = f"Structure A: {p['anchor']}\n\nStructure B: {p['candidate']}"
        texts.append(text)
        labels.append(float(p['label']))
        cosines.append(p['cosine_49d'])
    return Dataset.from_dict({'text': texts, 'label': labels, 'cosine_49d': cosines})

full_ds = pairs_to_dataset(pairs)
split = full_ds.train_test_split(test_size=0.2, seed=42)
train_ds = split['train']
eval_ds = split['test']

def tokenize_fn(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=MAX_LEN,
    )

train_tok = train_ds.map(tokenize_fn, batched=True, remove_columns=['text', 'cosine_49d'])
eval_tok = eval_ds.map(tokenize_fn, batched=True, remove_columns=['text', 'cosine_49d'])
train_tok.set_format('torch')
eval_tok.set_format('torch')

print(f'\u2705 データセット準備完了')
print(f'  訓練: {len(train_tok)} 件')
print(f'  評価: {len(eval_tok)} 件')

In [ ]:
# Cell 6: 訓練 Condition A (λ=0, L_Ξ なし)
from transformers import TrainingArguments, Trainer
from sklearn.metrics import mean_squared_error
import copy

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.squeeze()
    rho, p_val = spearmanr(preds, labels)
    mse = mean_squared_error(labels, preds)
    binary_preds = (preds > 0.5).astype(int)
    binary_labels = (labels > 0.5).astype(int)
    return {
        'rho': rho,
        'mse': mse,
        'accuracy': accuracy_score(binary_labels, binary_preds),
        'f1': f1_score(binary_labels, binary_preds, average='binary'),
    }

def make_args(run_name):
    return TrainingArguments(
        output_dir=f'/content/{run_name}',
        num_train_epochs=5,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        weight_decay=0.01,
        warmup_ratio=0.1,
        lr_scheduler_type='cosine',
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='rho',
        greater_is_better=True,
        bf16=True,
        logging_steps=10,
        report_to='none',
        remove_unused_columns=False,
        seed=42,
    )

# LoRA 初期重みを保存 (Condition B 用リセットに使う)
import copy
initial_lora_state = {}
for name, param in model.named_parameters():
    if 'lora' in name and param.requires_grad:
        initial_lora_state[name] = param.data.clone()

print('=' * 60)
print('Condition A: λ=0 (L_Ξ なし)')
print('=' * 60)

trainer_a = Trainer(
    model=model,
    args=make_args('phase_c_lambda0'),
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    compute_metrics=compute_metrics,
)

result_a = trainer_a.train()
eval_a = trainer_a.evaluate()

print(f'\n--- Condition A 結果 ---')
print(f'  ρ = {eval_a["eval_rho"]:.4f}')
print(f'  MSE = {eval_a["eval_mse"]:.4f}')
print(f'  F1 = {eval_a["eval_f1"]:.4f}')
print(f'  Accuracy = {eval_a["eval_accuracy"]:.4f}')

model.save_pretrained('/content/lora_lambda0')
print('\u2705 LoRA アダプタ保存: /content/lora_lambda0')

In [ ]:
# Cell 7: L_Ξ 正則化付き Trainer 定義
import torch.nn as nn

class XiRegularizedTrainer(Trainer):
    """Ξ 正則化: LoRA 重みの不均一度を最大化"""

    def __init__(self, *args, xi_lambda=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.xi_lambda = xi_lambda

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(**inputs)
        loss = outputs.loss

        if self.xi_lambda > 0:
            xi_total = 0.0
            n_params = 0
            for name, param in model.named_parameters():
                if 'lora' in name and param.requires_grad and param.numel() > 1:
                    xi_total += param.view(-1).float().var()
                    n_params += 1
            if n_params > 0:
                # -Ξ を加算 (Ξ 最大化 = 不均一忘却を促進)
                loss = loss - self.xi_lambda * (xi_total / n_params)

        return (loss, outputs) if return_outputs else loss

print('\u2705 XiRegularizedTrainer 定義完了')

In [ ]:
# Cell 8: 訓練 Condition B (λ=1.0, L_Ξ あり)

# LoRA 重みを初期状態にリセット
for name, param in model.named_parameters():
    if name in initial_lora_state:
        param.data.copy_(initial_lora_state[name])
print('=== LoRA 重みリセット完了 ===')

XI_LAMBDA = 1.0

print('=' * 60)
print(f'Condition B: λ={XI_LAMBDA} (L_Ξ あり)')
print('=' * 60)

trainer_b = XiRegularizedTrainer(
    model=model,
    args=make_args('phase_c_lambda1'),
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    compute_metrics=compute_metrics,
    xi_lambda=XI_LAMBDA,
)

result_b = trainer_b.train()
eval_b = trainer_b.evaluate()

print(f'\n--- Condition B (λ={XI_LAMBDA}) 結果 ---')
print(f'  ρ = {eval_b["eval_rho"]:.4f}')
print(f'  MSE = {eval_b["eval_mse"]:.4f}')
print(f'  F1 = {eval_b["eval_f1"]:.4f}')
print(f'  Accuracy = {eval_b["eval_accuracy"]:.4f}')

model.save_pretrained('/content/lora_lambda1')
print(f'\u2705 LoRA アダプタ保存: /content/lora_lambda1')

In [ ]:
# Cell 9: Spearman ρ + 偏ρ 評価 (Phase C-mini との比較)

def full_evaluation(model, pairs, n_sample=200, tag=''):
    """Spearman ρ + 偏ρ (コード長除去) + 分類指標"""
    model.eval()
    rng = np.random.RandomState(42)
    indices = rng.choice(len(pairs), min(n_sample, len(pairs)), replace=False)
    preds, labels, lengths = [], [], []

    with torch.no_grad():
        for i, idx in enumerate(indices):
            p = pairs[idx]
            inputs = encode_pair(p['anchor'], p['candidate'])
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            out = model(**inputs)
            preds.append(out.logits.squeeze().item())
            labels.append(p['label'])
            lengths.append(len(p['anchor']) + len(p['candidate']))
            if (i + 1) % 50 == 0:
                print(f'  [{i+1}/{len(indices)}]')

    preds = np.array(preds)
    labels = np.array(labels)
    lengths = np.array(lengths)

    rho, p_val = spearmanr(preds, labels)

    # 偏ρ (コード長の影響を除去)
    rho_pl, _ = spearmanr(preds, lengths)
    rho_ll, _ = spearmanr(labels, lengths)
    denom = np.sqrt(max(1 - rho_pl**2, 1e-10)) * np.sqrt(max(1 - rho_ll**2, 1e-10))
    partial_rho = (rho - rho_pl * rho_ll) / denom

    binary_preds = (preds > np.median(preds)).astype(int)
    binary_labels = (labels > 0.5).astype(int)

    print(f'\n{"=" * 50}')
    print(f'{tag} 評価結果')
    print(f'{"=" * 50}')
    print(f'  Spearman ρ      = {rho:.4f} (p = {p_val:.2e})')
    print(f'  偏ρ (コード長除去) = {partial_rho:.4f}')
    print(f'  F1              = {f1_score(binary_labels, binary_preds):.4f}')
    print(f'  Accuracy        = {accuracy_score(binary_labels, binary_preds):.4f}')
    return {'rho': rho, 'partial_rho': partial_rho, 'p_val': p_val}

print('=== Condition B (最終モデル) で評価 ===')
result_eval = full_evaluation(model, pairs, tag='Condition B')

In [ ]:
# Cell 10: 結果比較テーブル + 保存
import json, shutil

rho_a = eval_a.get('eval_rho', 0)
rho_b = eval_b.get('eval_rho', 0)
f1_a = eval_a.get('eval_f1', 0)
f1_b = eval_b.get('eval_f1', 0)
acc_a = eval_a.get('eval_accuracy', 0)
acc_b = eval_b.get('eval_accuracy', 0)

print('=' * 60)
print('Phase C QLoRA 結果比較')
print('=' * 60)
print(f'{"":30s} {"ρ":>8s} {"F1":>8s} {"Acc":>8s}')
print('-' * 60)
print(f'{"Phase B2 (Attentive Probe)":30s} {0.745:8.4f} {"N/A":>8s} {"N/A":>8s}')
print(f'{"Phase C-mini (CodeBERT)":30s} {0.963:8.4f} {"N/A":>8s} {"N/A":>8s}')
print(f'{"Baseline (pre-finetune)":30s} {baseline_rho:8.4f} {"N/A":>8s} {"N/A":>8s}')
print(f'{"Condition A (λ=0)":30s} {rho_a:8.4f} {f1_a:8.4f} {acc_a:8.4f}')
print(f'{"Condition B (λ=1.0)":30s} {rho_b:8.4f} {f1_b:8.4f} {acc_b:8.4f}')
print('-' * 60)
print(f'\nΔρ (A vs baseline): {rho_a - baseline_rho:+.4f}')
print(f'Δρ (B vs A):        {rho_b - rho_a:+.4f}')
print(f'Δρ (B vs C-mini):   {rho_b - 0.963:+.4f}')

print(f'\n--- 仮説判定 ---')
print(f'P11\' (QLoRA > C-mini): {"\u2705 支持" if rho_b > 0.963 else "\u274c 未達"} (Δρ = {rho_b - 0.963:+.4f})')
print(f'P14  (λ>0 > λ=0):    {"\u2705 支持" if rho_b > rho_a else "\u274c 未達"} (Δρ = {rho_b - rho_a:+.4f})')

summary = {
    'experiment': 'Phase C QLoRA',
    'model': MODEL_NAME,
    'n_pairs': len(pairs),
    'baseline_rho': baseline_rho,
    'condition_a': {'lambda': 0.0, 'rho': rho_a, 'f1': f1_a, 'accuracy': acc_a},
    'condition_b': {'lambda': XI_LAMBDA, 'rho': rho_b, 'f1': f1_b, 'accuracy': acc_b},
    'partial_rho_b': result_eval.get('partial_rho', None),
    'comparison': {'phase_c_mini_codebert': 0.963, 'phase_b2_probe': 0.745},
}

with open('/content/phase_c_qlora_results.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# Drive にも保存
drive_out = '/content/drive/MyDrive/lethe/'
shutil.copy('/content/phase_c_qlora_results.json', drive_out)
print(f'\n\u2705 結果保存: {drive_out}phase_c_qlora_results.json')

In [ ]:
# Cell 11: LoRA アダプタを Drive に保存
import shutil

# 最良条件のアダプタを Drive にコピー
best = 'lambda1' if rho_b >= rho_a else 'lambda0'
src_dir = f'/content/lora_{best}'
dst_dir = '/content/drive/MyDrive/lethe/lora_codellama13b_best'

shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
tokenizer.save_pretrained(dst_dir)
print(f'\u2705 Best adapter ({best}) Drive 保存: {dst_dir}')

shutil.make_archive('/content/lora_codellama13b_best', 'zip', src_dir)
shutil.copy('/content/lora_codellama13b_best.zip', '/content/drive/MyDrive/lethe/')
!ls -lh /content/lora_codellama13b_best.zip
print(f'\u2705 zip 保存: /content/drive/MyDrive/lethe/lora_codellama13b_best.zip')